In [36]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [37]:
x,y=load_diabetes(return_X_y=True)

In [38]:
x.shape

(442, 10)

In [39]:
y.shape

(442,)

In [40]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=2)

In [41]:
reg=LinearRegression()
reg.fit(x_train,y_train)

LinearRegression()

In [42]:
reg.coef_

array([  -9.15865318, -205.45432163,  516.69374454,  340.61999905,
       -895.5520019 ,  561.22067904,  153.89310954,  126.73139688,
        861.12700152,   52.42112238])

In [43]:
reg.intercept_

np.float64(151.88331005254167)

In [44]:
y_pred=reg.predict(x_test)
r2_score(y_test,y_pred)

0.4399338661568968

In [45]:
import numpy as np

class BatchGDRegressor:

    def __init__(self, learning_rate=0.01, epochs=100):
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.coef_ = None
        self.intercept_ = None

    def fit(self, x_train, y_train):
        self.intercept_ = 0
        self.coef_ = np.ones(x_train.shape[1])

        for i in range(self.epochs):
            y_hat = np.dot(x_train, self.coef_) + self.intercept_

            # Gradient for intercept
            intercept_der = -2 * np.mean(y_train - y_hat)
            self.intercept_ -= self.learning_rate * intercept_der

            # Gradient for coefficients
            coef_der = -2 * np.dot((y_train - y_hat), x_train) / x_train.shape[0]
            self.coef_ -= self.learning_rate * coef_der

        print("Intercept:", self.intercept_)
        print("Coefficients:", self.coef_)

    def predict(self, x_test):
        return np.dot(x_test, self.coef_) + self.intercept_


In [46]:
gd=BatchGDRegressor(epochs=1000,learning_rate=0.5)

In [47]:
gd.fit(x_train,y_train)

Intercept: 152.01351687661833
Coefficients: [  14.38990585 -173.7235727   491.54898524  323.91524824  -39.32648042
 -116.01061213 -194.04077415  103.38135565  451.63448787   97.57218278]


In [48]:
class SGDRegressor:

    def __init__(self, learning_rate=0.01, epochs=100):
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.coef_ = None
        self.intercept_ = None

    def fit(self, x_train, y_train):
        self.intercept_ = 0
        self.coef_ = np.ones(x_train.shape[1])

        for i in range(self.epochs):
            for j in range(x_train.shape[0]):
              idx=np.random.randint(0,x_train.shape[0])
              y_hat = np.dot(x_train[idx], self.coef_) + self.intercept_
              intercept_der = -2 * (y_train[idx] - y_hat)
              self.intercept_ = self.intercept_ - (self.learning_rate * intercept_der)
              coef_der = -2 * np.dot((y_train[idx] - y_hat), x_train[idx])
              self.coef_ = self.coef_ - (self.learning_rate * coef_der)

        print("Intercept:", self.intercept_)
        print("Coefficients:", self.coef_)

    def predict(self, x_test):
        return np.dot(x_test, self.coef_) + self.intercept_


In [49]:
sgd=SGDRegressor(epochs=50,learning_rate=0.01)

In [50]:
sgd.fit(x_train,y_train)

Intercept: 152.32932862632848
Coefficients: [  47.3434484   -65.73401429  349.64108546  251.3151686    12.82268393
  -34.41481513 -174.77558492  125.99691167  327.33580555  131.11684341]


In [51]:
y_pred=sgd.predict(x_test)
r2_score(y_test,y_pred)

0.43476124729360666

In [52]:
import numpy as np

class MiniBatchGD:

    def __init__(self, batch_size, learning_rate=0.01, epochs=100):
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.coef_ = None
        self.intercept_ = None

    def fit(self, x_train, y_train):

        self.intercept_ = 0
        self.coef_ = np.ones(x_train.shape[1])

        n_samples = x_train.shape[0]

        for i in range(self.epochs):

            for j in range(n_samples // self.batch_size):

                idx = np.random.choice(n_samples, self.batch_size, replace=False)

                x_batch = x_train[idx]
                y_batch = y_train[idx]

                y_hat = np.dot(x_batch, self.coef_) + self.intercept_

                # intercept gradient
                intercept_der = -2 * np.mean(y_batch - y_hat)
                self.intercept_ -= self.learning_rate * intercept_der

                # coefficient gradient
                coef_der = -2 * np.dot((y_batch - y_hat), x_batch) / self.batch_size
                self.coef_ -= self.learning_rate * coef_der

        print("Intercept:", self.intercept_)
        print("Coefficients:", self.coef_)

    def predict(self, x_test):
        return np.dot(x_test, self.coef_) + self.intercept_


In [59]:
bgd=MiniBatchGD(batch_size=int(x_train.shape[0]/50),epochs=100,learning_rate=0.1)

In [60]:
bgd.fit(x_train,y_train)

Intercept: 152.44816070667153
Coefficients: [  21.3046797  -182.77234732  481.75390552  316.82378134  -38.88724225
 -114.41696976 -188.46258895  105.79828386  447.44375661   85.18865609]


In [61]:
y_pred=bgd.predict(x_test)
r2_score(y_test,y_pred)

0.4554284830168759